# Anomaly Detection with the Denoising Autoencoder

The autoencoder was trained on FIDS30 (fruit images). We now test reconstruction error on three groups:

- **Normal (in-distribution):** FIDS30 test set — images it should reconstruct well
- **Near-OOD:** Nuts from Fruits-360 (almonds, cashews, pistachios) — visually similar to fruits
- **Far-OOD:** UC Merced satellite imagery — completely different domain

We expect: `Normal < Near-OOD < Far-OOD` in reconstruction MSE. If the autoencoder captures real fruit semantics, near-OOD should still be clearly separable from normal.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, roc_curve

IMG_SIZE = 128
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

## 1. Load the Trained Autoencoder

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = ConvAutoencoder().to(device)
model.load_state_dict(torch.load("denoising_ae.pth", map_location=device))
model.eval()
print("Autoencoder loaded.")

## 2. Load All Three Datasets

In [ ]:
normal_ds = datasets.ImageFolder("PrepData/FIDS30/Test", transform=transform)
near_ood_ds = datasets.ImageFolder("external_data/NearOOD", transform=transform)
far_ood_ds = datasets.ImageFolder("external_data/UC_Merced", transform=transform)

normal_loader = DataLoader(normal_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
near_ood_loader = DataLoader(near_ood_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
far_ood_loader = DataLoader(far_ood_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Normal   (FIDS30 test): {len(normal_ds)} images")
print(f"Near-OOD (nuts):        {len(near_ood_ds)} images")
print(f"Far-OOD  (UC Merced):   {len(far_ood_ds)} images")

## 3. Compute Reconstruction Error

Per-sample MSE between input and reconstruction. No noise added — we want to see
how well the autoencoder can reconstruct clean images it has never seen.

In [ ]:
def compute_recon_errors(model, loader):
    """Compute per-sample reconstruction MSE."""
    errors, all_images, all_recons = [], [], []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            recon = model(images)
            mse = (images - recon).pow(2).mean(dim=[1, 2, 3])
            errors.append(mse.cpu())
            all_images.append(images.cpu())
            all_recons.append(recon.cpu())
    return torch.cat(errors), torch.cat(all_images), torch.cat(all_recons)

normal_errors, normal_images, normal_recons = compute_recon_errors(model, normal_loader)
near_errors, near_images, near_recons = compute_recon_errors(model, near_ood_loader)
far_errors, far_images, far_recons = compute_recon_errors(model, far_ood_loader)

print(f"Normal   (FIDS30):    Mean MSE = {normal_errors.mean():.6f} ± {normal_errors.std():.6f}")
print(f"Near-OOD (nuts):      Mean MSE = {near_errors.mean():.6f} ± {near_errors.std():.6f}")
print(f"Far-OOD  (UC Merced): Mean MSE = {far_errors.mean():.6f} ± {far_errors.std():.6f}")
print(f"\nRatio near/normal: {near_errors.mean() / normal_errors.mean():.2f}x")
print(f"Ratio far/normal:  {far_errors.mean() / normal_errors.mean():.2f}x")

## 4. Reconstruction Error Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_errors.numpy(), bins=30, alpha=0.6,
        label=f"Normal / FIDS30 (n={len(normal_errors)})", color='steelblue')
ax.hist(near_errors.numpy(), bins=30, alpha=0.6,
        label=f"Near-OOD / nuts (n={len(near_errors)})", color='gold')
ax.hist(far_errors.numpy(), bins=30, alpha=0.6,
        label=f"Far-OOD / UC Merced (n={len(far_errors)})", color='tomato')
ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Count")
ax.set_title("Anomaly Detection: Reconstruction Error Distribution")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. ROC Curves and AUC

Two separate binary tasks, using reconstruction error as anomaly score:
- **Near-OOD vs Normal:** Can we separate nuts from fruits?
- **Far-OOD vs Normal:** Can we separate satellite images from fruits?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for ood_errors, name, color in [(near_errors, "Near-OOD (nuts)", "gold"),
                                  (far_errors, "Far-OOD (satellite)", "tomato")]:
    scores = torch.cat([normal_errors, ood_errors]).numpy()
    labels = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(ood_errors))])
    auc = roc_auc_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name}: AUC = {auc:.3f}")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label="Random")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="Anomaly Detection ROC Curves")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visual Comparison: Reconstructions

Show input → reconstruction for both normal and anomalous images.
The autoencoder should reconstruct fruits well but struggle with satellite images.

In [ ]:
def show_recon_comparison(images, recons, errors, title, n=6):
    """Show original, reconstruction, and difference for representative samples."""
    fig, axes = plt.subplots(3, n, figsize=(2.5 * n, 7.5))
    sorted_idx = errors.argsort()
    pick = sorted_idx[np.linspace(0, len(sorted_idx)-1, n, dtype=int)]

    for j, idx in enumerate(pick):
        inp = images[idx].permute(1, 2, 0).clamp(0, 1)
        rec = recons[idx].permute(1, 2, 0).clamp(0, 1)
        diff = (inp - rec).abs()

        axes[0, j].imshow(inp);  axes[0, j].axis('off')
        axes[1, j].imshow(rec);  axes[1, j].axis('off')
        axes[2, j].imshow(diff); axes[2, j].axis('off')
        axes[2, j].set_xlabel(f"MSE={errors[idx]:.4f}", fontsize=8)

    axes[0, 0].set_ylabel("Input", fontsize=11)
    axes[1, 0].set_ylabel("Reconstruction", fontsize=11)
    axes[2, 0].set_ylabel("Difference", fontsize=11)
    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

show_recon_comparison(normal_images, normal_recons, normal_errors,
                      "Normal (FIDS30) — Reconstruction Quality")
show_recon_comparison(near_images, near_recons, near_errors,
                      "Near-OOD (nuts) — Reconstruction Quality")
show_recon_comparison(far_images, far_recons, far_errors,
                      "Far-OOD (UC Merced) — Reconstruction Quality")